# Benchmarking to MODFLOW

This notebook provides a comparison of concentrations calculated with the *mibitrans* package and concentrations calculated with the numerical model *MODFLOW 2005*$^{1}$. The notebook benchmarks *mibitrans* showing that equation implementations are correct. 

**Authors:** Jorrit Bakker, Alraune Zech


## Background

We make use of the *flopy* Python-API for MODFLOW*$^{1}$ that solve the groundwater flow equation numerically and the transport equation MT3DMS package of the MODFLOW family. Calculations are based on the finite difference method. For the same choice of initial and boundary conditions, both the model class `Mibitrans` of the *mibitrans* package and MODFLOW should provide the same results.

**References**

$^1$ Harbaugh, A. W., MODFLOW-2005, the U.S. geological survey modular ground-water model – the ground-water flow process, Tech. Rep. 6-A16, U.S. Geological Survey Techniques and Methods, 2005.

## NOTE TO USER:

Running this notebook requires the additional dependency of having the executables for running MODFLOW-2005 and MT3DMS installed. See the [flopy-documentation](https://flopy.readthedocs.io/en/latest/md/get_modflow.html) for more information. Paths to the executables have been set to default values in the cell below, but should be manually set to the executable location on your machine if required.

In [ ]:
# Set the paths to executable locations for MODFLOW and MT3DMS
EXE_NAME_MF = 'mf2005'
EXE_NAME_MS = 'mt3dms'

In [ ]:
import flopy as fp
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt
from mibitrans.analysis.to_modflow import MibitransToModflow

## Benchmarking 

### Example dataset

Example values used in this comparison are those used for the Keesler model (see `example_keesler.ipynb` for more information on the example parameters), except for a zero decay rate.


In [ ]:
hydro = mbt.HydrologicalParameters(
    h_conductivity = 9.504, # [m/d]
    h_gradient = 0.003, # [m/m]
    porosity = 0.3, # [-]
    alpha_x = 10, # [m] (Note; high dispersivity values as using same input Keesler example)
    alpha_y = 1, # [m]
    alpha_z = 0 # [m]
)
att = mbt.AttenuationParameters(
    bulk_density=1700,             # [kg/m^3]
    partition_coefficient=0.038,   # [m^3/kg] (units cancel out with bulk density)
    fraction_organic_carbon=0.000057, # [-]
    decay_rate = 0,                # [1/day]
)
source = mbt.SourceParameters(# Note; as plume is symmetric, source is described from center outwards in y-direction.
    source_zone_boundary=np.array([2, 11, 20]), # [m]
    source_zone_concentration=np.array([13.68, 2.508, 0.057]), #[g/m3]
    depth=3, # [m]
    total_mass=np.inf, # [g]
)
model = mbt.ModelParameters(
    model_length=150, # [m]
    model_width=60, # [m]
    model_time=6*365, # [days]
    dx=1, # [m]
    dy=1, # [m]
    dt=365/5 # [days]
)

### Running mibitrans-model

In [ ]:
mbt_model = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_model.run()

### Converting *mibitrans*-input to MODLFOW-input

The `to_modflow` module of *mibitrans* allows to convert *mibitrans* dataclass input parameters into input for a combined MODFLOW-2005 and MT3DMS model.

The `MibitransToModflow()` class gives an object with flopy input as attributes that can be adapted (e.g. use `object.top` to change/check the depth of the model). Using `object.to_modflow()` creates flopy `mf` and `mt` objects, containing the input parameters. Consequently, any adaptations or additions to model can be performed on these objects. In the example below, the Btn package is overwritten to include additional output times for concentrations.

In [ ]:
mbt_mf = MibitransToModflow(hydro, att, source, model)
mbt_mf.to_modflow()

# Set the executables for MODFLOW and MT3DMS to user provided paths (from first cell in this notebook)
mbt_mf.exe_name_mf = EXE_NAME_MF
mbt_mf.exe_name_ms = EXE_NAME_MS

mf = mbt_mf.mf
mt = mbt_mf.mt
# Note: the line below changes the mt object in place, thus, it adapts the mt object inside the mbt_mf object.
mt3d_btn = fp.mt3d.Mt3dBtn(
    mt,
    icbund=mbt_mf.icbund,
    prsity=mbt_mf.prsity,
    sconc=mbt_mf.sconc,
    nprs=4,
    timprs=[365, 2*365,4*365,6*365]
)

### Running MODFLOW Model

Use the `run_modflow()` method to run the model, which returns the concentration distribution and output times. (this step requires the MODFLOW-2005 and MT3DMS executables). Instead, input can be written to MODFLOW input files using `mf.write_input()` and/or `mt.write_input()` and be used elsewhere.

In [ ]:
conc, times = mbt_mf.run_modflow()

Use the `centerline_modflow()` method to quickly visualize the calculated concentration distribution over the plume centerline for the last timestep.

In [ ]:
mbt_mf.centerline_modflow()
plt.show()

### Model comparison

The streamlined model setup allows simple comparison of both, the analytical and the numerical model results (for the same set of input parameters and flow/transport settings).



In [ ]:
# Determining location plume for plotting
x_pos = 80 # x-position index for transverse plot
plot_times = [1,2,4,6] # at which years to plot
x_mbt = mbt_results.x
y_mbt = mbt_results.y
model_middle = len(y_mbt)//2
row = mf.modelgrid.nrow // 2
x_mf = mf.modelgrid.xcellcenters[row, :]
y_mf = mf.modelgrid.ycellcenters[:, 0]

# Plot colors
cmap = plt.get_cmap('tab20b')
colors = cmap.colors
ci = 4
co = 2


fig, ax = plt.subplots(figsize=(10, 4), ncols=2)
for i, t in enumerate(plot_times):
    ax[0].plot(x_mbt, mbt_model.cxyt[t*5-1, model_middle, :], color=colors[ci*i+co], lw=3, label=f"t={t}y", alpha=0.3)
    ax[1].plot(y_mbt, mbt_model.cxyt[t*5-1, :, x_pos], color=colors[ci*i+co], lw=3, alpha=0.3)
for i, t in enumerate(plot_times):
    ax[0].plot(x_mf, conc[i, 0, row, :], linestyle="--", color=colors[ci*i+co], lw=3, label=" ")
    ax[1].plot(y_mf-60, conc[i, 0, :, x_pos], linestyle="--", color=colors[ci*i+co], lw=3,)
ax[1].set_xlim((-40,40))
ax[0].legend(title="   mibitrans    MODFLOW", ncol=2)
ax[0].set_xlabel("Distance from source [m]")
ax[1].set_xlabel("Distance from plume center [m]")
ax[0].set_ylabel(r"Concentration [$g/m^3$]")
plt.savefig('mibitrans_modflow.pdf')
plt.show()

Visual comparison of output from *mibitrans* and MODFLOW indicates nearly identical results - as is expected. 

The only noteable difference is near the model boundaries. Model boundaries are only relevant for the analytical models regarding the visualization domain. For the numerical model constant head boundary conditions introduce a slight deviation. Picking a model extent larger than the relevant plume area would prevent these minor differences.

### Quantify differences


In [ ]:
###  Functions for quantitative comparison
from mibitrans.analysis import differences as diff

In [ ]:
# max_source_conc = bio_source.source_zone_concentration[0]

# mad_bio_bio_nd = diff.mean_absolute_difference(
#     bioscreen_data.nodecay / max_source_conc, bio_results_comp_nodecay.relative_cxyt[bio_time_index,:,:]
# )

# mad_bio_bio_lindec = diff.mean_absolute_difference(
#     bioscreen_data.lineardecay / max_source_conc, bio_results_comp_linear.relative_cxyt[bio_time_index,:,:]
# )

# mad_bio_bio_instant = diff.mean_absolute_difference(
#     bioscreen_data.instant / max_source_conc, bio_results_comp_instant.relative_cxyt[bio_time_index,:,:]
# )

In [ ]:
# print("Mean absolute difference in relative concentration between MODFLOW modl "
#       "and Mibitrans model class:",mad_bat_mibi_nd)
